# Solutions · Chapter 05-03 · Multiple regression

Worked answers for `notebooks/05_regression/05-03_multiple_linear.ipynb`.

**E9, E11 and E20 are the three to read**: one shows that irrelevant columns are almost harmless, one
fixes collinearity by engineering rather than deleting, and one asks what a library does when the problem
has no unique answer at all.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

# SYNTHETIC: the chapter's 300 houses. TRUTH: +3.0 per m2, +0.9 per year.
rng = np.random.default_rng(53)
n_houses = 300
age = rng.uniform(0, 80, n_houses)
size = 130 - 0.7 * age + rng.normal(0, 10, n_houses)
price = 40 + 3.0 * size + 0.9 * age + rng.normal(0, 20, n_houses)
houses = pd.DataFrame({"size_m2": size, "age_years": age, "price_k": price})

BOTH = LinearRegression().fit(houses[["size_m2", "age_years"]], houses.price_k)
print("two-feature fit: size %+.4f, age %+.4f   (truth: +3.0, +0.9)"
      % (BOTH.coef_[0], BOTH.coef_[1]))

## Quick understanding

### E1

**The change in the target associated with a one-unit change in that feature, among rows that are
otherwise similar in the other columns of the model** - estimated from whatever such comparisons the data
happens to contain.

The three qualifications all matter: *associated with*, not *caused by*; *the other columns of the model*,
not all other variables; and *the comparisons the data contains*, not a hypothetical intervention.

### E2

1. Regress the **feature** on the controls, and keep the residuals - the part of the feature the controls
   cannot explain.
2. Regress the **target** on the controls, and keep those residuals too.
3. Fit a simple regression of the second set of residuals on the first. **Its slope is the
   multiple-regression coefficient.**

### E3

**Damages:** the individual coefficients - their size, their stability, sometimes their sign - and
therefore any statement of the form "this feature is worth X".

**Leaves alone:** the predictions, the residuals, and R-squared. In the chapter's experiment the
coefficient spread grew twenty-three-fold while the prediction spread did not move at all.

## Hand calculation

### E4

The omitted-variable formula: `marginal = direct + (effect of the omitted) × (slope of the omitted on the
included)`.

For **age alone**, the omitted variable is size, whose direct effect is 3.0, and the slope of size on age
is **-0.6619**:

`0.9 + 3.0 × (-0.6619)` = **-1.0857**, against a measured **-1.0304**.

The two differ by 0.05 because the formula uses the *true* direct effects while the fit uses estimates
from 300 noisy rows - but it predicts the sign flip and nearly the magnitude, from arithmetic done before
any regression was run.

For **size alone**, the omitted variable is age (effect 0.9) and the slope of age on size is -1.0634:
`3.0 + 0.9 × (-1.0634)` = **2.0429**, against a measured **1.9242**.

### E5

`VIF = 1 / (1 - r²) = 1 / (1 - 0.81)` = **5.2632**.

The standard error is inflated by the **square root**, so the coefficient is **2.29 times noisier** than
it would be if the two features were independent. Equivalently: you would need about 5.3 times as much
data to get the same precision.

### E6

`3.0 thousand per m²` ÷ `10.764 sq ft per m²` = **0.27871 thousand per square foot.**

**The model did not change at all.** Same fit, same predictions, same residuals, same R-squared. Only the
number changed, because the number carries a unit - 05-02's E7 again, and the reason a coefficient's
magnitude means nothing on its own.

### E7

**"A one standard deviation increase in income is associated with a 0.62 standard deviation increase in
the target, holding age constant"** - and the same for age at 0.11.

To convert back you need **the standard deviations of the original columns**: multiply the standardised
coefficient by `sd(target) / sd(feature)`. Without those two numbers the coefficients are comparable to
each other but cannot be stated in currency or years.

In [ ]:
slope_size_on_age = LinearRegression().fit(houses[["age_years"]], houses.size_m2).coef_[0]
slope_age_on_size = LinearRegression().fit(houses[["size_m2"]], houses.age_years).coef_[0]

print("E4  slope of size on age = %.4f" % slope_size_on_age)
print("    predicted age-alone coefficient  = 0.9 + 3.0 x %.4f = %+.4f   (measured -1.0304)"
      % (slope_size_on_age, 0.9 + 3.0 * slope_size_on_age))
print("    predicted size-alone coefficient = 3.0 + 0.9 x %.4f = %+.4f   (measured +1.9242)"
      % (slope_age_on_size, 3.0 + 0.9 * slope_age_on_size))
print()
print("E5  VIF = 1 / (1 - 0.9^2) = %.4f, so the standard error is %.2f times wider"
      % (1 / (1 - 0.81), np.sqrt(1 / (1 - 0.81))))
print("E6  3.0 per m2 = %.5f per square foot" % (3.0 / 10.764))

## Coding

### E8 - Frisch-Waugh-Lovell as a function

In [ ]:
# fit `target` on `feature`, having removed `controls` from both
def partial_coefficient(frame, target, feature, controls):
    feature_left = frame[feature] - LinearRegression().fit(
        frame[controls], frame[feature]).predict(frame[controls])
    target_left = frame[target] - LinearRegression().fit(
        frame[controls], frame[target]).predict(frame[controls])
    return float(LinearRegression().fit(feature_left.to_numpy().reshape(-1, 1), target_left).coef_[0])


for feature, controls in [("size_m2", ["age_years"]), ("age_years", ["size_m2"])]:
    direct = BOTH.coef_[["size_m2", "age_years"].index(feature)]
    partial = partial_coefficient(houses, "price_k", feature, controls)
    print("%-10s  multiple regression %.6f   partialled out %.6f   match %s"
          % (feature, direct, partial, np.isclose(direct, partial)))

### E9 - what do irrelevant columns cost?

In [ ]:
rows = []
for extra in [0, 1, 5, 20, 100]:
    noise_rng = np.random.default_rng(7)
    design = houses[["size_m2", "age_years"]].copy()
    for column in range(extra):
        design["noise_%d" % column] = noise_rng.normal(size=n_houses)
    fitted = LinearRegression().fit(design, houses.price_k)
    rows.append({"noise columns": extra,
                 "size coefficient": round(float(fitted.coef_[0]), 4),
                 "age coefficient": round(float(fitted.coef_[1]), 4),
                 "R2 on the fitting data": round(fitted.score(design, houses.price_k), 4)})
print(pd.DataFrame(rows).to_string(index=False))

**One noise column moves the coefficients by less than 0.003. Twenty move them by about 0.03.** Against a
size coefficient of 2.80, that is a shift of one percent.

**Irrelevant columns are nearly harmless to the coefficients, and that is worth contrasting with the
chapter.** Collinearity - columns that are *related to the ones you care about* - inflated the spread
twenty-three-fold. Pure noise, being uncorrelated with everything, barely registers.

What noise columns *do* damage is the last column: **R-squared rises monotonically as they are added**,
because each new column gives the fit one more way to chase the residuals. That is why R-squared on the
fitting data cannot be used to choose between models with different numbers of features, and it is
04-05's "adding a column that cannot help still lowers the training error" arriving in a regression
setting.

At 100 noise columns on 300 rows the effect becomes severe - which is 05-07's subject.

### E10 - the trade-off curve

In [ ]:
def spreads(correlation, draws=300, rows=300, seed=1):
    generator = np.random.default_rng(seed)
    first = generator.normal(size=rows)
    second = correlation * first + np.sqrt(max(1e-9, 1 - correlation ** 2)) * generator.normal(size=rows)
    target = 2 * first + 2 * second + generator.normal(0, 1, rows)
    design = np.column_stack([first, second])
    coefficients, predictions = [], []
    for _ in range(draws):
        picked = generator.integers(0, rows, rows)
        refit = LinearRegression().fit(design[picked], target[picked])
        coefficients.append(refit.coef_)
        predictions.append(refit.predict(design[:20]))
    return (float(np.array(coefficients)[:, 0].std()),
            float(np.array(predictions).std(axis=0).mean()))


correlations = [0.0, 0.5, 0.9, 0.99, 0.999]
measured = [spreads(c) for c in correlations]

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.plot(correlations, [m[0] for m in measured], "o-", color="#D55E00", linewidth=2.2,
        markersize=8, label="spread of the coefficient")
ax.plot(correlations, [m[1] for m in measured], "s-", color="#0072B2", linewidth=2.2,
        markersize=8, label="spread of the predictions")
ax.set_yscale("log")
ax.set_xlabel("correlation between the two features")
ax.set_ylabel("standard deviation across 300 refits (log scale)")
ax.set_title("One of these explodes and the other is a flat line", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(pd.DataFrame({"correlation": correlations,
                    "coefficient spread": [round(m[0], 4) for m in measured],
                    "prediction spread": [round(m[1], 4) for m in measured]}).to_string(index=False))

**The orange line rises by more than an order of magnitude; the blue one is flat.** On a log axis the
contrast is unmissable, and it is the single most useful thing to know about collinearity.

### E11 - engineering the problem away

In [ ]:
districts = fetch_california_housing(as_frame=True).frame.sample(3000, random_state=0)
districts = districts.assign(bedroom_share=districts.AveBedrms / districts.AveRooms)


def inflation_factors(frame, columns):
    return {column: round(1 / (1 - LinearRegression().fit(
        frame[[c for c in columns if c != column]], frame[column]).score(
        frame[[c for c in columns if c != column]], frame[column])), 2) for column in columns}


for label, columns in [("original", ["AveRooms", "AveBedrms"]),
                       ("engineered", ["AveRooms", "bedroom_share"])]:
    fitted = LinearRegression().fit(districts[columns], districts.MedHouseVal)
    print("%-11s %s" % (label, "  ".join("%s %+.4f" % (c, v) for c, v in zip(columns, fitted.coef_))))
    print("%-11s inflation factors: %s" % ("", inflation_factors(districts, columns)))
    print("%-11s R2: %.4f" % ("", fitted.score(districts[columns], districts.MedHouseVal)))
    print()

**The inflation factor falls from 2.92 to 1.27, and both coefficients become readable.**

- **`AveRooms` +0.0278**: each extra room per household adds a little to the district's median value.
- **`bedroom_share` -3.6900**: a district where a larger *fraction* of rooms are bedrooms is cheaper -
  smaller rooms, denser occupancy, less living space per room.

Compare that with the original pair, `+0.2901` and `-1.4405`, which required a careful sentence about
holding one constant while varying the other. **The engineered columns say the same thing more directly,
because they are closer to being independent.**

**And it cost something, which I did not expect and should not hide.** R-squared falls from **0.1091 to
0.0479** - the engineered pair predicts less than half as well.

The reason is worth understanding, because it limits how far this trick goes. Replacing `AveBedrms` with
`AveBedrms / AveRooms` is a **nonlinear** transformation, so the two pairs do not span the same space: a
linear model on `(AveRooms, AveBedrms)` can represent any combination of rooms and bedrooms, and one on
`(AveRooms, bedroom_share)` cannot. Some genuine signal lived in that difference and has been discarded.

So this is a real trade rather than a free win:

| | original pair | engineered pair |
|---|---|---|
| inflation factor | 2.92 | 1.27 |
| coefficients | need a careful sentence | read directly |
| R-squared | 0.1091 | 0.0479 |

**Which to prefer depends on the question**, which is the chapter's thesis arriving with a price tag
attached. For a report explaining what drives district values, the engineered pair is clearer and the
accuracy is not the point. For a model that predicts, keep the original pair and do not read its
coefficients. Fitting both and saying so remains the honest third option.

One further hazard: the ratio is undefined if `AveRooms` is zero and unstable when it is small, so a real
pipeline would need a guard.

### Can you get the interpretability without paying the accuracy?

A **linear** reparameterisation spans the identical space, so it cannot cost any R-squared. But it does
not automatically reduce collinearity either - it depends entirely which one you pick.

In [ ]:
districts = districts.assign(
    rooms_minus_bedrooms=districts.AveBedrms - districts.AveRooms,
    bedrooms_beyond_rooms=districts.AveBedrms - LinearRegression().fit(
        districts[["AveRooms"]], districts.AveBedrms).predict(districts[["AveRooms"]]))

for label, columns in [("as recorded", ["AveRooms", "AveBedrms"]),
                       ("a difference", ["AveRooms", "rooms_minus_bedrooms"]),
                       ("orthogonalised", ["AveRooms", "bedrooms_beyond_rooms"]),
                       ("the ratio (nonlinear)", ["AveRooms", "bedroom_share"])]:
    fitted = LinearRegression().fit(districts[columns], districts.MedHouseVal)
    print("%-24s R2 %.4f   VIF %6.2f   coefficients %s"
          % (label, fitted.score(districts[columns], districts.MedHouseVal),
             list(inflation_factors(districts, columns).values())[0],
             np.round(fitted.coef_, 4)))

**Three linear reparameterisations, all with identical R-squared, and inflation factors of 2.92, 62.14 and
1.00.**

- **The difference `AveBedrms - AveRooms` makes it far worse.** `AveRooms` is much larger and more
  variable than `AveBedrms`, so the difference is dominated by `-AveRooms` and is therefore *more*
  correlated with it than `AveBedrms` was.
- **Orthogonalising** - replacing `AveBedrms` with the part of it that `AveRooms` cannot predict - gives
  an inflation factor of exactly **1.00**, by construction.

And look at the coefficient it produces for `AveRooms`: **+0.0735**, which is precisely the
*single-feature* value from the chapter. That is not a coincidence, and it is the whole idea in one
number: once the second column carries none of the first, the first's coefficient is free to be its own
marginal effect.

**So the honest summary of E11:** if you want interpretability *and* accuracy, orthogonalise - it is
free. If you want a column that means something specific to a domain expert, like "the share of rooms
that are bedrooms", you may pay for it, and here that cost was more than half the R-squared.

### E12 - units and standardising

In [ ]:
standardised_size = (houses.size_m2 - houses.size_m2.mean()) / houses.size_m2.std()

raw_fit = LinearRegression().fit(houses[["size_m2", "age_years"]], houses.price_k)
scaled_fit = LinearRegression().fit(np.column_stack([standardised_size, houses.age_years]),
                                    houses.price_k)

print("size coefficient, raw          : %.4f  (per m2)" % raw_fit.coef_[0])
print("size coefficient, standardised : %.4f  (per standard deviation)" % scaled_fit.coef_[0])
print("their ratio                    : %.4f" % (scaled_fit.coef_[0] / raw_fit.coef_[0]))
print("the standard deviation of size : %.4f" % houses.size_m2.std())
print()
print("predictions identical:", bool(np.allclose(
    raw_fit.predict(houses[["size_m2", "age_years"]]),
    scaled_fit.predict(np.column_stack([standardised_size, houses.age_years])))))

**The standardised coefficient is the raw one multiplied by the feature's standard deviation** -
`2.7977 × 18.0295 = 50.4403` - and the predictions are bit-identical.

That is the exact conversion, and it makes the trade explicit. The raw coefficient answers *"what is a
square metre worth?"*, which is actionable. The standardised one answers *"how much does this feature
move the target, compared with the others?"*, which is comparable. **Neither is more correct; report the
one that matches the question**, and never compare raw coefficients across features on different scales.

## Interpretation

### E13

**It can be correct, and it usually is.**

`years_experience` and `age` are strongly correlated - most people accumulate both at the same rate - so
the model is comparing people of the **same experience but different ages**. Someone who is 45 with ten
years of experience entered the field later than someone who is 32 with ten years, and in most professions
the later entrant earns less: fewer years in *this* career, a career change, or a break.

So the negative coefficient reads: **"among people with the same experience, being older is associated
with slightly lower pay."** That is a real and interesting finding about late entrants and career breaks,
and it is completely invisible in a model with only one of the two columns.

What it does **not** mean is that ageing lowers pay. And with two features this correlated, the split
between them is exactly the unstable quantity the chapter measured - so the number should be reported with
its inflation factor, and probably alongside a model using `years_experience` alone.

### E14

**The coefficient is not an effect, and closing the emergency route would not shorten stays.**

The regression compares patients admitted through emergency with patients admitted otherwise, *holding the
model's other columns constant*. Emergency admissions differ from planned ones in a hundred ways the model
does not contain - the condition, its urgency, whether it was elective surgery with a scheduled recovery
period. Many planned admissions are precisely the ones with long scheduled stays.

**The -1.3 says "emergency patients happen to stay 1.3 days less than otherwise-similar patients", not
"routing a patient through emergency shortens their stay by 1.3 days".** The route is a marker of what
kind of patient they are.

The general form, and it is the most consequential misreading in applied regression: **a coefficient
becomes an effect only when the comparison it makes is the comparison the intervention would make.** Here
it is not - closing the route does not turn those patients into planned admissions, it turns them into
emergency patients arriving somewhere else. 00-04 made this point with a campaign; it is the same point.

## Debugging

### E15

**1. Are the two features correlated?** Compute the correlation and the inflation factor. A large shift in
one coefficient when another is added is the *expected* behaviour when the two overlap - the omitted-
variable formula in E4 predicts both the direction and roughly the size. If they are correlated, nothing
is broken and the question becomes which model answers your question.

**2. Is the new feature partly a function of the target, or of the future?** A shift this large also
happens when the added column is leaky - 04-05's `months_on_file` reduced every other coefficient to
irrelevance. Check when the new column's value becomes knowable.

In that order, because the first is one line and explains most cases.

### E16

**The cause is near-perfect collinearity** - two or more columns that are almost exact linear combinations
of each other. The model is estimating an enormous positive coefficient on one and an enormous negative
one on another, and they cancel in the prediction. That is why the predictions are fine: the chapter's
diagonal cloud, taken to its extreme.

**Two fixes:**

1. **Find and remove the redundancy.** Compute the inflation factors; a value in the hundreds or thousands
   points at the culprit. Often it is an accident - a column included both raw and as a percentage, a
   one-hot encoding without a dropped level, or a total alongside all of its parts.
2. **Regularise.** Ridge regression (05-09) adds a penalty on coefficient size, which resolves the
   ambiguity by preferring the small-and-balanced solution over the huge-and-cancelling one. It is the
   standard answer when the redundancy is real and you want to keep both columns.

## Exam and interview reasoning

### E17

> "It is the change in the target associated with a one-unit change in that feature, comparing rows that
> are similar in the model's other columns. The precise version is that if you remove everything the other
> columns explain - from that feature and from the target - and fit a line to what is left, its slope is
> the coefficient. So a coefficient is not a property of the feature; it is a property of the feature given
> the rest of the model, and it changes when the model changes. In the data I was working with, the age
> coefficient was negative on its own and positive once house size was included, and both were correct
> answers to different questions."

**"So it tells us what would happen if we changed that variable?"**

> "Only if the comparison the regression made is the comparison the change would make. The regression
> compares houses that already differ in age while being similar in size; an intervention would take one
> specific house and change something about it. Those coincide when the data comes from an experiment, or
> when you can argue that nothing else systematically differs between the groups being compared. Otherwise
> the coefficient is a description of the data, not a prediction about an action - and the failure mode is
> concrete: a hospital model gave a negative coefficient on emergency admission, which certainly does not
> mean closing the emergency department shortens stays."

## Transfer to a different situation

### E18

**The fertiliser coefficient will mean: "among fields with the same measured soil nitrogen and the same
rainfall, those given more fertiliser yielded X more."**

And that comparison is close to meaningless as an effect, because **fertiliser was applied in response to
nitrogen.** The fields that got more fertiliser are the fields that were measured as deficient. So among
fields matched on *measured* nitrogen, the ones receiving more fertiliser are those the agronomist judged
to need it for reasons the model does not contain - drainage, previous crop, the parts of soil quality the
nitrogen test does not capture.

The coefficient will therefore be **biased downwards**, possibly to zero or negative, and a naive reading
would say fertiliser does not work.

**What you would need to interpret it as an effect:** ideally a **randomised trial** - assign fertiliser
rates at random to plots, which breaks the link between the treatment and the field's condition. Failing
that: an instrument (something that shifts fertiliser use but is unrelated to yield, such as a price change
or a subsidy boundary), or historical variation in application that was *not* driven by the soil.

This is the general shape of the problem, and it has a name: **the treatment was assigned on the basis of
information related to the outcome.** It is the single most common reason an observational coefficient is
not an effect, and no amount of controlling for observed columns fixes it - because the deciding
information was not recorded.

## Explain it to someone non-technical

### E19

> "Older houses in this town are mostly small cottages, and newer ones are mostly large. So if you just
> compare old houses with new ones, the old ones are cheaper - but that is mostly because they are smaller,
> not because they are old. If instead you compare two houses of *the same size*, one older and one newer,
> the older one is worth more - people pay for character. Both statements describe the same houses. They
> just answer different questions: 'what do old houses cost?' and 'what is age worth?'"

(89 words.)

## Optional challenge

### E20 - what happens when there is no unique answer?

In [ ]:
noise_rng = np.random.default_rng(7)
extended = houses[["size_m2", "age_years"]].copy()
extended["noise"] = noise_rng.normal(size=n_houses)
three = LinearRegression().fit(extended, houses.price_k)

print("three-feature fit:", {c: round(float(v), 6) for c, v in zip(extended.columns, three.coef_)})
print()
for feature in extended.columns:
    controls = [c for c in extended.columns if c != feature]
    frame = extended.assign(price_k=houses.price_k)
    print("  %-10s multiple %.6f   partialled out %.6f"
          % (feature, three.coef_[list(extended.columns).index(feature)],
             partial_coefficient(frame, "price_k", feature, controls)))

In [ ]:
# now make two columns perfectly collinear: the second is exactly twice the first
perfect = np.column_stack([houses.size_m2, houses.size_m2 * 2.0])
degenerate = LinearRegression().fit(perfect, houses.price_k)
single = LinearRegression().fit(houses[["size_m2"]], houses.price_k)

print("coefficients returned for (size, 2 x size): %s" % np.round(degenerate.coef_, 6))
print("their combined effect per m2: %.6f + 2 x %.6f = %.6f"
      % (degenerate.coef_[0], degenerate.coef_[1],
         degenerate.coef_[0] + 2 * degenerate.coef_[1]))
print("the single-feature coefficient              : %.6f" % single.coef_[0])
print()
print("predictions identical to the one-feature model:",
      bool(np.allclose(degenerate.predict(perfect), single.predict(houses[["size_m2"]]))))
print("sum of squared coefficients returned        : %.6f" % (degenerate.coef_ ** 2).sum())
print("an equally valid alternative, (1.9242, 0)   : %.6f" % (single.coef_[0] ** 2))

**Every coefficient is reproduced exactly by partialling out**, including the noise column - the
Frisch-Waugh-Lovell result holds however many controls there are.

**And the degenerate case is the interesting one.** With `size` and `2 × size` in the same model there are
infinitely many correct answers: any pair `(a, b)` with `a + 2b = 1.9242` fits identically. The library
does not error - it returns **`(0.3848, 0.7697)`**, whose combined effect is exactly 1.9242, and whose
predictions are bit-identical to the one-feature model.

Why that pair? Because `LinearRegression` solves with a pseudo-inverse, which among all exact solutions
returns the one with the **smallest sum of squared coefficients**. Here that is 0.7405 against 3.7025 for
the equally-valid `(1.9242, 0)`. **The answer it gives is as good as any other and no better** - it is a
tie-break, not a finding.

Three things worth taking from that:

- **A coefficient can be entirely an artefact of the solver.** Nothing in the data preferred 0.3848 over
  1.9242. If you had read that coefficient as "a square metre is worth 0.38", you would have been reporting
  a numerical convention.
- **It does not warn you.** No exception, no warning, a perfect fit and a sensible-looking pair of numbers.
  The only signal is an inflation factor that has gone to infinity, which is why computing them is worth
  the one line.
- **The predictions remain exactly right**, which is the chapter's thesis at its limit: even when the
  coefficients are completely undetermined, the model predicts as well as it possibly could.

That minimum-norm tie-break is also, not coincidentally, what ridge regression does deliberately and
continuously rather than only in the degenerate case - which is where 05-09 picks this up.